# Jude 2026b · intended speech T17

## Prepare the data

Run the commands below from the repository root with `bci-repro` active.

### 1. Download

Download `Data.zip` from [Dryad](https://doi.org/10.5061/dryad.vq83bk481). Create `studies/20260428_jude/data/raw/dryad_vq83bk481/` and save the download as `studies/20260428_jude/data/raw/dryad_vq83bk481/dryad_vq83bk481__Data.zip`.

### 2. Extract

Extract the archive into `studies/20260428_jude/data/extracted/dryad_vq83bk481/`.

```text
python -m zipfile -e "studies/20260428_jude/data/raw/dryad_vq83bk481/dryad_vq83bk481__Data.zip" "studies/20260428_jude/data/extracted/dryad_vq83bk481"
```

After extraction, check for `studies/20260428_jude/data/extracted/dryad_vq83bk481/Data/t17/`. The session folders contain `brainToText/RedisMat/*.mat`.

### 3. Prepare the model input

Convert the configured sentence blocks from RedisMat `.mat` files to the HDF5 layout used by the trainer.

```text
python shared/tools/setup_upstream.py --study 20260428_jude
python studies/20260428_jude/code/prepare_data.py
```

**Output** `studies/20260428_jude/data/derived/dryad_vq83bk481/hdf5/trainer_input/`. Each session has `data_train.hdf5` and/or `data_val.hdf5`. The session index is `studies/20260428_jude/data/derived/dryad_vq83bk481/hdf5/dryad_vq83bk481__session_description.csv`.

## Read the archived results

The offline model used the last eligible sentence block for validation. Figure 3a pools the greedy phoneme and post-LM word errors from that block by recording day.

This walkthrough uses the checked-in archive and takes no data download.

In [1]:
import csv
import json
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "figures/reproduction_benchmark.yaml").is_file())
sys.path.insert(0, str(root / "figures/code"))
from figure_paths import PUBLISHED, load_benchmark

benchmark = load_benchmark()

def read_rows(path):
    """Read the archived CSV rows in their original order."""
    with (root / path).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

## From a trial or day to the reported result

In [2]:
arm = "Jude 2026 (LIS, T17)"
from build_session_per_vs_wer import build, read_tsv, word_ed

source = benchmark["figure3_per_sources"][arm]
rows = read_rows(source["greedy"])
decoded = read_tsv(root / source["decode"])
trial = rows[0]
reference, hypothesis = decoded[0]
print(f"{trial['session']} · {reference}")
print(f"Word hypothesis: {hypothesis}")
edits, words = word_ed(reference, hypothesis)
print(f"Trial word edits: {edits}/{words}")
result = build(arm, root / source["greedy"], root / source["decode"])
day = result["sessions"][0]
print(f"{day['session']}: {day['n']} trials, PER {day['per']:.2f}%, WER {day['wer']:.2f}%")
print(f"{len(result['sessions'])} sessions, {len(rows)} evaluated trials; pooled {result['pooled']}")

t17.2024.03.26 · They hope I have my nurse.
Word hypothesis: THEY HOPE YOU HAVE A NICE
Trial word edits: 3/6
Jude 2026 (LIS, T17): 12 sessions | pooled PER 39.97% | WER 46.01%
t17.2024.03.26: 40 trials, PER 27.12%, WER 38.04%
12 sessions, 417 evaluated trials; pooled {'per': 39.97, 'wer': 46.01}


## Continue the full reproduction

For a separate data disk, set `BCI_DATA_ROOT` to the directory that contains the study folders and use the same layout under that root.

See the [evaluation and input guide](docs/20261124_evaluation_protocol/README.md) and [code guide](code/README.md).

**Public commands** [`prepare_data.py`](code/prepare_data.py), [`train.py`](code/train.py), [`infer.py`](code/infer.py), [`evaluate.py`](code/evaluate.py), [`decode.py`](code/decode.py). Run a command with `--help` to see its arguments.